Demo of Caltech101 Step1


In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
from transformers import BertTokenizer, BertModel
import timm
import numpy as np
import gradio as gr
from google.colab import drive
from tqdm.auto import tqdm

# 1. SETUP & DRIVE
drive.mount('/content/drive')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_PATH = '/content/drive/MyDrive/caltech101_vit_base_best.pth'

# 2. MODEL ARCHITECTURE (ViT-Base)
print("Initializing ViT-Base (AugReg) Architecture...")
# Base model has 768 hidden dimensions
image_model = timm.create_model('vit_base_patch16_224.augreg_in21k_ft_in1k', pretrained=True).to(device)
image_model.reset_classifier(0)

# Since ViT-Base output is 768 and BERT is 768, this is a 768->768 mapping
projection = nn.Linear(768, 768).to(device)
logit_scale = nn.Parameter(torch.tensor(np.log(1 / 0.07), device=device))

# 3. LOAD SAVED WEIGHTS (If they exist)
if os.path.exists(SAVE_PATH):
    print(f"📦 Loading existing ViT-Base weights from {SAVE_PATH}...")
    checkpoint = torch.load(SAVE_PATH, map_location=device)
    image_model.load_state_dict(checkpoint['image_model_state_dict'])
    projection.load_state_dict(checkpoint['projection_state_dict'])
    logit_scale.data = checkpoint['logit_scale']
    classes = checkpoint['classes']
    SHOULD_TRAIN = False
    print(f"✅ Loaded! Previous Accuracy: {checkpoint.get('accuracy', 0):.2f}%")
else:
    print("❌ No weights found. Preparing for training...")
    SHOULD_TRAIN = True

# 4. DATASET & TRANSFORM (Always needed for Demo or Training)
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.Lambda(lambda x: x.convert('RGB') if x.mode != 'RGB' else x),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

dataset = torchvision.datasets.Caltech101(root='./data', download=True, transform=transform)
classes = [c.replace('_', ' ') for c in dataset.categories]

# 5. BERT TEXT EMBEDDINGS (Always needed)
print("Generating BERT text embeddings...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert = BertModel.from_pretrained('bert-base-uncased').to(device)
bert.eval()

def get_text_features(cls_list):
    phrases = [f"a photo of a {c}" for c in cls_list]
    inputs = tokenizer(phrases, padding=True, return_tensors='pt').to(device)
    with torch.no_grad():
        out = bert(**inputs)
    return F.normalize(out.last_hidden_state[:, 0, :], dim=-1)

text_embeddings = get_text_features(classes)
del bert # Save memory

Mounted at /content/drive
Initializing ViT-Base (AugReg) Architecture...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

📦 Loading existing ViT-Base weights from /content/drive/MyDrive/caltech101_vit_base_best.pth...
✅ Loaded! Previous Accuracy: 98.04%


100%|██████████| 137M/137M [00:03<00:00, 34.4MB/s]


...

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# --- Always create the loaders so metrics/demo can run ---
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

# Set a seed so the split is the same every time you reopen the file
torch.manual_seed(42)
train_set, test_set = random_split(dataset, [train_size, test_size])

# Create the loader (use num_workers=0 to avoid the Colab AssertionError)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False, num_workers=0)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=0)

print(f"✅ DataLoaders initialized: {len(test_set)} images ready for testing.")

✅ DataLoaders initialized: 1736 images ready for testing.


In [ ]:


# 6. TRAINING SECTION (Only runs if weights are missing)
if SHOULD_TRAIN:
    train_size = int(0.8 * len(dataset))
    train_set, test_set = random_split(dataset, [train_size, len(dataset)-train_size])
    train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=32)

    optimizer = torch.optim.AdamW([
        {'params': image_model.parameters(), 'lr': 1e-5}, # Slower LR for Base model
        {'params': projection.parameters(), 'lr': 5e-5}
    ], weight_decay=0.01)

    best_acc = 0
    for epoch in range(10): # ViT-Base converges fast
        image_model.train(); projection.train()
        for imgs, lbls in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            feat = F.normalize(projection(image_model(imgs)), dim=-1)
            logits = (logit_scale.exp() * feat @ text_embeddings.T)
            loss = F.cross_entropy(logits, lbls)
            loss.backward()
            optimizer.step()

        # Validation
        image_model.eval(); projection.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, lbls in test_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                feat = F.normalize(projection(image_model(imgs)), dim=-1)
                acc = (feat @ text_embeddings.T).argmax(dim=-1)
                correct += (acc == lbls).sum().item()
                total += lbls.size(0)

        current_acc = 100 * correct / total
        print(f"Epoch {epoch+1} Accuracy: {current_acc:.2f}%")

        if current_acc > best_acc:
            best_acc = current_acc
            torch.save({
                'image_model_state_dict': image_model.state_dict(),
                'projection_state_dict': projection.state_dict(),
                'logit_scale': logit_scale.data,
                'classes': classes,
                'accuracy': best_acc
            }, SAVE_PATH)



Epoch 1:   0%|          | 0/217 [00:00<?, ?it/s]

Epoch 1 Accuracy: 82.43%


Epoch 2:   0%|          | 0/217 [00:00<?, ?it/s]

Epoch 2 Accuracy: 93.72%


Epoch 3:   0%|          | 0/217 [00:00<?, ?it/s]

...

Epoch 8 Accuracy: 98.04%


Epoch 9:   0%|          | 0/217 [00:00<?, ?it/s]

Epoch 9 Accuracy: 97.98%


Epoch 10:   0%|          | 0/217 [00:00<?, ?it/s]

Epoch 10 Accuracy: 97.70%


In [ ]:
# 3. LOAD SAVED WEIGHTS (If they exist)
if os.path.exists(SAVE_PATH):
    print(f"📦 Loading existing ViT-Base weights from {SAVE_PATH}...")
    checkpoint = torch.load(SAVE_PATH, map_location=device)
    image_model.load_state_dict(checkpoint['image_model_state_dict'])
    projection.load_state_dict(checkpoint['projection_state_dict'])
    logit_scale.data = checkpoint['logit_scale']
    classes = checkpoint['classes']
    SHOULD_TRAIN = False
    print(f"✅ Loaded! Previous Accuracy: {checkpoint.get('accuracy', 0):.2f}%")
else:
    print("❌ No weights found. Preparing for training...")
    SHOULD_TRAIN = True

📦 Loading existing ViT-Base weights from /content/drive/MyDrive/caltech101_vit_base_best.pth...
✅ Loaded! Previous Accuracy: 98.04%


In [ ]:
import gradio as gr
import torch.nn.functional as F

# 1. We need the BERT tokenizer and model again to encode custom labels
from transformers import BertTokenizer, BertModel
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_text_model = BertModel.from_pretrained('bert-base-uncased').to(device)
bert_text_model.eval()

def predict_zero_shot(img, custom_labels):
    if img is None or not custom_labels:
        return "Please upload an image and enter labels."

    # --- 1. Process the Image ---
    img_t = transform(img).unsqueeze(0).to(device)

    # --- 2. Process the Custom Labels ---
    # Split the comma-separated string into a list
    label_list = [l.strip() for l in custom_labels.split(",")]
    # Add a prompt template for better performance
    prompts = [f"a photo of a {l}" for l in label_list]

    # Encode the new labels using BERT
    inputs = tokenizer(prompts, padding=True, truncation=True, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = bert_text_model(**inputs)
        # Use the [CLS] token and normalize
        custom_text_embeddings = F.normalize(outputs.last_hidden_state[:, 0, :], dim=-1)

    # --- 3. Run Inference ---
    image_model.eval()
    projection.eval()

    with torch.no_grad():
        # Visual Embedding (ViT Base)
        features = image_model(img_t)
        image_features = F.normalize(projection(features), dim=-1)

        # Calculate similarity (Zero-Shot)
        logits = (logit_scale.exp() * image_features @ custom_text_embeddings.T)
        probs = logits.softmax(dim=-1).cpu().numpy()[0]

    # Return results as a dictionary for Gradio Label component
    return {label_list[i]: float(probs[i]) for i in range(len(label_list))}

# --- 4. Launch the Zero-Shot Interface ---
gr.Interface(
    fn=predict_zero_shot,
    inputs=[
        gr.Image(type="pil", label="Upload Image"),
        gr.Textbox(
            label="Custom Labels (comma separated)",
            placeholder="e.g. cat, dog, spaceship, pizza",
            value="cat, dog, car, tree" # default starting values
        )
    ],
    outputs=gr.Label(num_top_classes=5),
    title="🌍 True Zero-Shot Classifier",
    description="Type any labels you want! The model will compare the image to your text on the fly using ViT-Base and BERT."
).launch(share=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7747b98640ba8ef113.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import time
import torch
import numpy as np
from tqdm.auto import tqdm

def benchmark_and_report():
    image_model.eval()
    projection.eval()

    # --- 1. GPU Latency & Throughput Benchmark ---
    print("🚀 Benchmarking GPU Performance...")
    # Use a standard 224x224 input
    dummy_input = torch.randn(1, 3, 224, 224).to(device)

    # Warm-up
    for _ in range(10):
        with torch.no_grad():
            _ = projection(image_model(dummy_input))

    torch.cuda.synchronize()
    start_time = time.time()

    num_iterations = 100
    with torch.no_grad():
        for _ in range(num_iterations):
            _ = projection(image_model(dummy_input))

    torch.cuda.synchronize()
    end_time = time.time()

    latency_ms = ((end_time - start_time) / num_iterations) * 1000
    throughput_fps = 1000 / latency_ms

    # --- 2. Accuracy Collection ---
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for imgs, lbls in tqdm(test_loader, desc="Gathering Accuracy"):
            imgs = imgs.to(device)
            feat = torch.nn.functional.normalize(projection(image_model(imgs)), dim=-1)
            preds = (feat @ text_embeddings.T).argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(lbls.numpy())

    # --- 3. Final Calculations ---
    acc = accuracy_score(all_labels, all_preds)
    # Now that this is imported, it will work!
    p_m, r_m, f_m, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro')

    # ViT-Base Theoretical GFLOPs
    total_gflops = 35.2

    print("\n" + "📊" + "="*45)
    print(f"{'VIT-BASE HARDWARE & ACCURACY REPORT':^45}")
    print("="*47)
    print(f"Overall Accuracy:       {acc*100:.2f}%")
    print(f"Macro Accuracy:         {r_m*100:.2f}%")
    print("-" * 47)
    print(f"Latency (GPU):          {latency_ms:.2f} ms / image")
    print(f"Throughput (FPS):       {throughput_fps:.2f} frames/sec")
    print(f"Total GFLOPs:           {total_gflops:.2f}")
    print(f"Accuracy per GFLOP:     {acc/total_gflops:.4f}")
    print("="*47)

# Run the fixed report
benchmark_and_report()

🚀 Benchmarking GPU Performance...


Gathering Accuracy:   0%|          | 0/55 [00:00<?, ?it/s]


📊=============================================
     VIT-BASE HARDWARE & ACCURACY REPORT     
Overall Accuracy:       99.60%
Macro Accuracy:         99.47%
-----------------------------------------------
Latency (GPU):          14.05 ms / image
Throughput (FPS):       71.16 frames/sec
Total GFLOPs:           35.20
Accuracy per GFLOP:     0.0283
